In [5]:
path = os.path.join(MSI_INPUT_FOLDER, MSI_SAMPLE_FILES[0])
adata_var = sc.read_h5ad(path, backed='r').var
adata_var

,mzs,mzs_original
326.1934,326.1934,326.1907
337.1867,337.1867,337.1835
339.2243,339.2243,339.2211
340.2283,340.2283,340.2250
341.2403,341.2403,341.2370
...,...,...
1001.6889,1001.6889,1001.6712
1002.6961,1002.6913,1002.6784
1003.7043,1003.7043,1003.6866
1017.6613,1017.6613,1017.6436


In [6]:
import scanpy as sc
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# Configuration
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]

# 1. Load the Reference (The "Gold Standard")
ref_path = os.path.join(MSI_INPUT_FOLDER, MSI_SAMPLE_FILES[0])
adata_ref = sc.read_h5ad(ref_path)
target_features = adata_ref.var_names
print(f"Reference established from {MSI_SAMPLE_FILES[0]} with {len(target_features)} features.")

synced_adatas = {MSI_SAMPLE_FILES[0]: adata_ref}

# 2. Synchronize all other animals to the Reference
for file in MSI_SAMPLE_FILES[1:]:
    path = os.path.join(MSI_INPUT_FOLDER, file)
    adata = sc.read_h5ad(path)
    
    # Identify missing features (present in Ref but not in current file)
    missing_features = target_features.difference(adata.var_names)
    
    if len(missing_features) > 0:
        print(f"Adding {len(missing_features)} missing features to {file}...")
        
        # Create a zero-filled sparse matrix for the missing columns
        zero_data = csr_matrix((adata.n_obs, len(missing_features)))
        
        # Create a temporary AnnData for missing features
        adata_missing = sc.AnnData(
            X=zero_data,
            obs=adata.obs,
            var=pd.DataFrame(index=missing_features)
        )
        
        # Merge (concatenate) the original data with the new zero-filled columns
        import anndata as ad
        adata = ad.concat([adata, adata_missing], axis=1, merge="same")

    # 3. Final alignment: Reorder and subset to match the reference exactly
    # This removes any features the current animal had that the Ref did NOT have
    adata = adata[:, target_features].copy()
    
    synced_adatas[file] = adata
    print(f"Successfully synced {file}. Shape: {adata.shape}")

# Verification Check
all_match = all(list(synced_adatas[f].var_names == target_features) for f in MSI_SAMPLE_FILES)
print(f"\n✅ All 16 animals now have identical variable names and order: {all_match}")

Reference established from halfbrain_yc_1_filtered_common.h5ad with 528 features.
Adding 516 missing features to halfbrain_yc_2_filtered_common.h5ad...
Successfully synced halfbrain_yc_2_filtered_common.h5ad. Shape: (7858, 528)
Adding 335 missing features to halfbrain_yc_3_filtered_common.h5ad...
Successfully synced halfbrain_yc_3_filtered_common.h5ad. Shape: (7150, 528)
Adding 522 missing features to halfbrain_yc_4_filtered_common.h5ad...
Successfully synced halfbrain_yc_4_filtered_common.h5ad. Shape: (6067, 528)
Adding 519 missing features to halfbrain_yad_1_filtered_common.h5ad...
Successfully synced halfbrain_yad_1_filtered_common.h5ad. Shape: (7517, 528)
Adding 518 missing features to halfbrain_yad_2_filtered_common.h5ad...
Successfully synced halfbrain_yad_2_filtered_common.h5ad. Shape: (7596, 528)
Adding 519 missing features to halfbrain_yad_3_filtered_common.h5ad...
Successfully synced halfbrain_yad_3_filtered_common.h5ad. Shape: (6844, 528)
Adding 518 missing features to halfb

In [12]:
import scanpy as sc
import os
import numpy as np
import pandas as pd

# --- Configuration ---
MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]

# Tolerance for matching m/z values (adjust based on your instrument resolution)
MZ_TOLERANCE = 0.02 

# 1. Load Reference (Animal 1)
ref_path = os.path.join(MSI_INPUT_FOLDER, MSI_SAMPLE_FILES[0])
adata_ref = sc.read_h5ad(ref_path)
# Convert index to numeric m/z values for calculation
ref_mzs = adata_ref.var_names.astype(float) 

print(f"Reference: {MSI_SAMPLE_FILES[0]} ({len(ref_mzs)} features)")

# 2. Process other animals
for file_name in MSI_SAMPLE_FILES[1:]:
    path = os.path.join(MSI_INPUT_FOLDER, file_name)
    adata = sc.read_h5ad(path)
    current_mzs = adata.var_names.astype(float)
    
    new_names = []
    matches_found = 0
    
    # Compare each m/z in the current animal to the reference
    for mz in current_mzs:
        # Find the absolute difference between current mz and all reference mzs
        diffs = np.abs(ref_mzs - mz)
        closest_idx = np.argmin(diffs)
        min_diff = diffs[closest_idx]
        
        # If the closest reference mz is within tolerance, rename it to the reference name
        if min_diff <= MZ_TOLERANCE:
            new_names.append(adata_ref.var_names[closest_idx])
            matches_found += 1
        else:
            # If no close match is found, keep its original name (or handle as unique)
            new_names.append(str(mz))
            
    adata.var_names = new_names
    
    # Now that names match the reference, we filter to keep ONLY the reference features
    # This ensures the final matrix matches Animal 1 exactly
    common_to_ref = [name for name in new_names if name in adata_ref.var_names]
    adata = adata[:, common_to_ref].copy()
    
    print(f"Processed {file_name}: Matched {matches_found}/{len(current_mzs)} features to Reference.")
    
    # Save the synchronized file (optional)
    # adata.write(os.path.join(MSI_INPUT_FOLDER, "synced_" + file_name))

Reference: halfbrain_yc_1_filtered_common.h5ad (528 features)
Processed halfbrain_yc_2_filtered_common.h5ad: Matched 528/528 features to Reference.
Processed halfbrain_yc_3_filtered_common.h5ad: Matched 528/528 features to Reference.
Processed halfbrain_yc_4_filtered_common.h5ad: Matched 528/528 features to Reference.
Processed halfbrain_yad_1_filtered_common.h5ad: Matched 528/528 features to Reference.
Processed halfbrain_yad_2_filtered_common.h5ad: Matched 528/528 features to Reference.
Processed halfbrain_yad_3_filtered_common.h5ad: Matched 528/528 features to Reference.
Processed halfbrain_yad_4_filtered_common.h5ad: Matched 528/528 features to Reference.
Processed halfbrain_ac_1_filtered_common.h5ad: Matched 528/528 features to Reference.
Processed halfbrain_ac_2_filtered_common.h5ad: Matched 528/528 features to Reference.
Processed halfbrain_ac_3_filtered_common.h5ad: Matched 528/528 features to Reference.
Processed halfbrain_ac_4_filtered_common.h5ad: Matched 528/528 features to